In [ ]:
!pip install pandas numpy lightgbm shap matplotlib seaborn plotly prophet -q

import pandas as pd
import numpy as np
from pathlib import Path
import warnings
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor

warnings.filterwarnings('ignore')

print("Import thành công")

data_path = r"C:\DATATHON ROUND1"
folder = Path(data_path)

files = {
    'products.csv': {},
    'customers.csv': {'parse_dates': ['signup_date']},
    'promotions.csv': {'parse_dates': ['start_date', 'end_date']},
    'geography.csv': {},
    'orders.csv': {'parse_dates': ['order_date']},
    'order_items.csv': {},
    'payments.csv': {},
    'shipments.csv': {'parse_dates': ['ship_date', 'delivery_date']},
    'returns.csv': {'parse_dates': ['return_date']},
    'reviews.csv': {'parse_dates': ['review_date']},
    'sales.csv': {'parse_dates': ['Date']},
    'sample_submission.csv': {'parse_dates': ['Date']},
    'inventory.csv': {'parse_dates': ['snapshot_date']},
    'web_traffic.csv': {'parse_dates': ['date']},
}

data = {}
for filename, kwargs in files.items():
    df = pd.read_csv(folder / filename, **kwargs)
    data[filename.replace('.csv', '')] = df
    print(f" {filename:<20} | Shape: {df.shape}")

# ==============================================================================
# 1. LOAD & PREPARE DATA
# ==============================================================================
print("Đang tải dữ liệu...")
sales = pd.read_csv(folder / 'sales.csv', parse_dates=['Date']).rename(columns={'Date': 'date', 'Revenue': 'revenue', 'COGS': 'cogs'})
sample_sub = pd.read_csv(folder / 'sample_submission.csv', parse_dates=['Date'])
promos = pd.read_csACv(folder / 'promotions.csv', parse_dates=['start_date', 'end_date'])
sales = sales.set_index('date').sort_index()

AUP_PROXY = 6000 

# 2. FEATURE ENGINEERING:
def build_pure_features(date_index, promos_df):
    df = pd.DataFrame(index=date_index)
    
    # A. Thời gian nội tại & Mùa vụ
    df['dayofweek'] = df.index.dayofweek
    df['month'] = df.index.month
    df['quarter'] = df.index.quarter
    df['dayofmonth'] = df.index.day
    df['is_weekend'] = df['dayofweek'].isin([5, 6]).astype(int)
    df['is_month_start'] = df.index.is_month_start.astype(int)
    df['is_month_end'] = df.index.is_month_end.astype(int)
    
    for k in [1, 2, 3]:
        df[f'fourier_sin_{k}'] = np.sin(2 * np.pi * k * df.index.dayofyear / 365.25)
        df[f'fourier_cos_{k}'] = np.cos(2 * np.pi * k * df.index.dayofyear / 365.25)
        
    # B. Mã hóa drop 2018
    anchor_date = pd.to_datetime('2019-01-01')
    df['days_since_2019'] = (df.index - anchor_date).days
    df['days_since_2019'] = df['days_since_2019'].clip(lower=0)
    
    # C. Khuyến mãi 
    active_count = []
    promo_power = []
    
    for d in df.index:
        active = promos_df[(promos_df['start_date'] <= d) & (promos_df['end_date'] >= d)]
        active_count.append(len(active))
        
        if len(active) > 0:
            pct_sum = active[active['promo_type'] == 'percentage']['discount_value'].sum()
            fixed_sum = active[active['promo_type'] == 'fixed']['discount_value'].sum()
            fixed_as_pct = (fixed_sum / AUP_PROXY) * 100 
            power = pct_sum + fixed_as_pct
        else:
            power = 0
        promo_power.append(power)
        
    df['active_promos_count'] = active_count
    df['promo_power'] = promo_power
    df['power_x_weekend'] = df['promo_power'] * df['is_weekend']
    
    return df

print("Đang tạo đặc trưng nội sinh...")
X_full = build_pure_features(sales.index, promos)

mask = X_full.index >= '2019-01-01'
X_train = X_full[mask]
y_train_rev = sales.loc[mask, 'revenue']
y_train_cogs = sales.loc[mask, 'cogs']

# 3. KAGGLE ENSEMBLE
print("\nĐang huấn luyện các mô hình...")
lgb_rev = lgb.LGBMRegressor(objective='regression_l1', metric='mae', n_estimators=1800, learning_rate=0.015, num_leaves=31, subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1)
lgb_rev.fit(X_train, y_train_rev)

lgb_cogs = lgb.LGBMRegressor(objective='regression_l1', metric='mae', n_estimators=1500, learning_rate=0.015, num_leaves=25, subsample=0.8, random_state=42, n_jobs=-1)
lgb_cogs.fit(X_train, y_train_cogs)

xgb_rev = xgb.XGBRegressor(objective='reg:absoluteerror', eval_metric='mae', n_estimators=1200, learning_rate=0.015, max_depth=5, subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1)
xgb_rev.fit(X_train, y_train_rev)

xgb_cogs = xgb.XGBRegressor(objective='reg:absoluteerror', eval_metric='mae', n_estimators=1000, learning_rate=0.015, max_depth=5, subsample=0.8, random_state=42, n_jobs=-1)
xgb_cogs.fit(X_train, y_train_cogs)

cat_rev = CatBoostRegressor(loss_function='MAE', iterations=1500, learning_rate=0.02, depth=6, verbose=0, random_seed=42)
cat_rev.fit(X_train, y_train_rev)

cat_cogs = CatBoostRegressor(loss_function='MAE', iterations=1200, learning_rate=0.02, depth=5, verbose=0, random_seed=42)
cat_cogs.fit(X_train, y_train_cogs)


# 4. PREDICT & INTERNAL LOGIC BINDING
future_dates = pd.to_datetime(sample_sub['Date'])
X_test = build_pure_features(future_dates, promos)

pred_rev_lgb = lgb_rev.predict(X_test)
pred_rev_xgb = xgb_rev.predict(X_test)
pred_rev_cat = cat_rev.predict(X_test)

pred_cogs_lgb = lgb_cogs.predict(X_test)
pred_cogs_xgb = xgb_cogs.predict(X_test)
pred_cogs_cat = cat_cogs.predict(X_test)

pred_rev_final = (pred_rev_lgb * 0.4) + (pred_rev_xgb * 0.4) + (pred_rev_cat * 0.2)
pred_cogs_final = (pred_cogs_lgb * 0.4) + (pred_cogs_xgb * 0.4) + (pred_cogs_cat * 0.2)

pred_rev_final = np.maximum(pred_rev_final, 0)
pred_cogs_final = np.maximum(pred_cogs_final, 0)
pred_cogs_final = np.minimum(pred_cogs_final, pred_rev_final * 0.95)

threshold_power = np.percentile(X_full['promo_power'], 95)
for i in range(len(future_dates)):
    if X_test['promo_power'].iloc[i] >= threshold_power and threshold_power > 0:
        pred_rev_final[i] *= 1.25 
        pred_cogs_final[i] *= 1.25

# TẠO FILE VÀ TẢI XUỐNG
submission = sample_sub.copy()
submission['Revenue'] = pred_rev_final.round(2)
submission['COGS'] = pred_cogs_final.round(2)
file_name = 'submission_god_mode_v3.csv'
submission.to_csv(file_name, index=False)

from IPython.display import FileLink, HTML
display(HTML(f"### Tải dự đoán"))
display(FileLink(file_name, result_html_prefix="📥 Click tải file: "))